# 02 — USDA Food Ingestion Pipeline

Execution notebook for building the application-ready USDA Foundation Foods dataset.

Pipeline:

`USDA CSV → filter Foundation Foods → select application nutrients → join → pivot → normalize → validate → Parquet`

In [1]:
import sys
from pathlib import Path

# Make the src/ package importable when running the notebook directly.
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from ingest._00_config import FOUNDATION_DIR, PROCESSED_DIR
from ingest._01_filters import filter_foundation_foods
from ingest._02_nutrients import select_application_nutrients
from ingest._03_transform import join_foods_with_nutrients, pivot_nutrients
from ingest._04_validate import validate_foods
from ingest._05_export import export_foods, export_foods_csv

## 1. Start Spark

In [2]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("CalorieTrackerFoodIngestion")
    .getOrCreate()
)

spark.version

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/19 19:42:06 WARN Utils: Your hostname, sahil-IdeaPad-Gaming-3-15ACH6, resolves to a loopback address: 127.0.1.1; using 192.168.29.208 instead (on interface wlo1)
26/09/19 19:42:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/19 19:42:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'4.2.0'

## 2. Load USDA source files

Raw CSVs are intentionally loaded without schema inference. IDs and numeric measurements are cast explicitly inside the transformation functions.

In [3]:
def read_csv(filename: str):
    return (
        spark.read
        .option("header", True)
        .option("quote", '"')
        .option("escape", '"')
        .option("multiLine", True)
        .csv(str(FOUNDATION_DIR / filename))
    )


food_df = read_csv("food.csv")
food_nutrient_df = read_csv("food_nutrient.csv")
nutrient_df = read_csv("nutrient.csv")

print("food.csv:", food_df.count())
print("food_nutrient.csv:", food_nutrient_df.count())
print("nutrient.csv:", nutrient_df.count())

food.csv: 87990
food_nutrient.csv: 170469
nutrient.csv: 477


## 3. Filter canonical Foundation Foods

In [4]:
foundation_food_df = filter_foundation_foods(food_df)

print("Foundation Food records:", foundation_food_df.count())

foundation_food_df.select(
    "fdc_id",
    "description",
    "data_type",
).show(10, truncate=False)

Foundation Food records: 469
+------+--------------------------------------------------------------------------+---------------+
|fdc_id|description                                                               |data_type      |
+------+--------------------------------------------------------------------------+---------------+
|321358|Hummus, commercial                                                        |foundation_food|
|321359|Milk, reduced fat, fluid, 2% milkfat, with added vitamin A and vitamin D  |foundation_food|
|321360|Tomatoes, grape, raw                                                      |foundation_food|
|321505|Salt, table, iodized                                                      |foundation_food|
|321611|Beans, snap, green, canned, regular pack, drained solids                  |foundation_food|
|321900|Broccoli, raw                                                             |foundation_food|
|322228|Milk, lowfat, fluid, 1% milkfat, with added vitamin A and vitam

## 4. Select application nutrients

This uses the nutrient configuration in `src/ingest/config.py` as the application source of truth.

In [5]:
application_nutrients_df = select_application_nutrients(nutrient_df)

print("Application nutrients:", application_nutrients_df.count())

application_nutrients_df.select(
    "nutrient_id",
    "name",
    "unit_name",
    "field",
    "display_order",
).show(50, truncate=False)

Application nutrients: 32
+-----------+------------------------------+---------+----------------+-------------+
|nutrient_id|name                          |unit_name|field           |display_order|
+-----------+------------------------------+---------+----------------+-------------+
|1008       |Energy                        |KCAL     |calories        |1            |
|1003       |Protein                       |G        |protein         |2            |
|1005       |Carbohydrate, by difference   |G        |carbohydrates   |3            |
|1004       |Total lipid (fat)             |G        |fat             |4            |
|1079       |Fiber, total dietary          |G        |fiber           |5            |
|2000       |Total Sugars                  |G        |sugars          |6            |
|1258       |Fatty acids, total saturated  |G        |saturated_fat   |7            |
|1093       |Sodium, Na                    |MG       |sodium          |8            |
|1087       |Calcium, Ca    

## 5. Join foods with nutrient measurements

In [6]:
joined_df = join_foods_with_nutrients(
    foundation_food_df,
    food_nutrient_df,
    application_nutrients_df,
)

print("Joined rows:", joined_df.count())

joined_df.select(
    "fdc_id",
    "name",
    "nutrient_id",
    "field",
    "amount",
).show(20, truncate=False)

Joined rows: 7682
+------+--------------------------------------------------------------------------+-----------+-------+------+
|fdc_id|name                                                                      |nutrient_id|field  |amount|
+------+--------------------------------------------------------------------------+-----------+-------+------+
|321358|Hummus, commercial                                                        |1003       |protein|7.35  |
|321359|Milk, reduced fat, fluid, 2% milkfat, with added vitamin A and vitamin D  |1003       |protein|3.35  |
|321360|Tomatoes, grape, raw                                                      |1003       |protein|0.83  |
|321611|Beans, snap, green, canned, regular pack, drained solids                  |1003       |protein|1.04  |
|321900|Broccoli, raw                                                             |1003       |protein|2.57  |
|322228|Milk, lowfat, fluid, 1% milkfat, with added vitamin A and vitamin D       |1003       

## 6. Check duplicate nutrient measurements

The pivot currently uses `first(amount)`. This check makes duplicate measurements visible before the final dataset is produced.

In [7]:
duplicates_df = (
    joined_df
    .groupBy("fdc_id", "nutrient_id", "field")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicates_df.count()
print("Duplicate food/nutrient groups:", duplicate_count)

if duplicate_count:
    duplicates_df.show(50, truncate=False)

Duplicate food/nutrient groups: 1
+-------+-----------+---------+-----+
|fdc_id |nutrient_id|field    |count|
+-------+-----------+---------+-----+
|2768188|1106       |vitamin_a|2    |
+-------+-----------+---------+-----+



## 7. Pivot nutrients into one row per food

In [8]:
foods_df = pivot_nutrients(joined_df)

print("Final rows before validation:", foods_df.count())
print("Final columns:")
print(foods_df.columns)

foods_df.show(10)

Final rows before validation: 468
Final columns:
['fdc_id', 'name', 'calories', 'protein', 'carbohydrates', 'fat', 'fiber', 'sugars', 'saturated_fat', 'sodium', 'calcium', 'iron', 'magnesium', 'phosphorus', 'potassium', 'zinc', 'copper', 'manganese', 'selenium', 'vitamin_a', 'vitamin_c', 'vitamin_d', 'vitamin_e', 'vitamin_k', 'thiamin', 'riboflavin', 'niacin', 'pantothenic_acid', 'vitamin_b6', 'biotin', 'folate', 'vitamin_b12', 'caffeine', 'cholesterol']


26/09/19 19:42:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+--------------------+--------+--------+-------------+------+------+------+-------------+------+-------+------+---------+----------+---------+-------+-------+---------+--------+---------+---------+---------+---------+---------+-------+----------+------+----------------+----------+------+------+-----------+--------+-----------+
| fdc_id|                name|calories| protein|carbohydrates|   fat| fiber|sugars|saturated_fat|sodium|calcium|  iron|magnesium|phosphorus|potassium|   zinc| copper|manganese|selenium|vitamin_a|vitamin_c|vitamin_d|vitamin_e|vitamin_k|thiamin|riboflavin|niacin|pantothenic_acid|vitamin_b6|biotin|folate|vitamin_b12|caffeine|cholesterol|
+-------+--------------------+--------+--------+-------------+------+------+------+-------------+------+-------+------+---------+----------+---------+-------+-------+---------+--------+---------+---------+---------+---------+---------+-------+----------+------+----------------+----------+------+------+-----------+--------+--

## 8. Normalize carbohydrate-by-difference

USDA carbohydrate-by-difference values can occasionally be slightly negative because the value is calculated from measured components. Negative carbohydrate is not meaningful for the application, so normalize it to zero before validation.

In [9]:
foods_df = foods_df.withColumn(
    "carbohydrates",
    F.greatest(
        F.col("carbohydrates"),
        F.lit(0.0),
    ),
)

negative_carbs = foods_df.filter(
    F.col("carbohydrates") < 0
).count()

print("Negative carbohydrate values after normalization:", negative_carbs)

Negative carbohydrate values after normalization: 0


## 9. Validate final dataset

In [10]:
validate_foods(foods_df)
print("Validation passed.")

Validation passed.


## 10. Export application dataset

In [11]:
# 1. Define the output paths clearly
parquet_output = PROCESSED_DIR / "foods.parquet"
csv_output = PROCESSED_DIR / "foods.csv"

# 2. Run the export functions
export_foods(foods_df, parquet_output)
export_foods_csv(foods_df, csv_output)

# 3. Print the actual destinations (clarifying they are directories)
print(f"Exported Parquet directory to: {parquet_output}")
print(f"Exported CSV directory to:     {csv_output}")

Exported Parquet directory to: /home/sahil/Desktop/Calorie Counter/import/data/processed/foods.parquet
Exported CSV directory to:     /home/sahil/Desktop/Calorie Counter/import/data/processed/foods.csv


## 11. Final sanity check

In [12]:
print("Rows:", foods_df.count())
print("Columns:", len(foods_df.columns))

foods_df.select(
    "fdc_id",
    "name",
    "calories",
    "protein",
    "carbohydrates",
    "fat",
).show(10, truncate=False)

Rows: 468
Columns: 34
+-------+---------------------------------------+--------+--------+-------------+------+
|fdc_id |name                                   |calories|protein |carbohydrates|fat   |
+-------+---------------------------------------+--------+--------+-------------+------+
|324653 |Pickles, cucumber, dill or kosher dill |12.0    |0.48    |1.99         |0.43  |
|1750339|Apples, red delicious, with skin, raw  |NULL    |0.1875  |14.7817      |0.2125|
|325871 |Bread, white, commercially prepared    |270.0   |9.43    |49.2         |3.59  |
|327923 |Lettuce, cos or romaine, raw           |17.0    |1.24    |3.24         |0.26  |
|2261421|Flour, oat, whole grain                |NULL    |13.16875|69.91725     |6.309 |
|1750353|Spinach, mature                        |NULL    |2.90625 |2.64135      |0.6044|
|1750354|Tomato, roma                           |NULL    |0.695625|3.837475     |0.425 |
|2759001|Lunchmeat, turkey, oven roasted, sliced|NULL    |NULL    |0.0          |NULL  |

## 12. Stop Spark

Run this when the notebook session is finished.

In [13]:
spark.stop()